In [1]:
import re

with open('../the-verdict.txt', 'r', encoding='utf-8') as file:
    raw = file.read()

print(len(raw))
print(raw[0:100])

20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [2]:
word_array = re.split(r'([_,.!-();:?"\']|--|\s)', raw)
word_array = [item.lower() for item in word_array if item.strip()]
print(word_array[0:20])
print(len(word_array))

['i', 'had', 'always', 'thought', 'jack', 'gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was']
4690


In [3]:
sorted_words = sorted(set(word_array))
sorted_words.extend(["<unk>", "<endoftext>"])
vocab_size = len(sorted_words)
print(vocab_size)

1078


In [4]:
vocab = {token:idx for idx,token in enumerate(sorted_words)}

for idx, (key,value) in enumerate(vocab.items()):
    print(key, value)
    if idx >= 20:
        break

! 0
" 1
' 2
( 3
) 4
, 5
-- 6
. 7
: 8
; 9
? 10
_ 11
a 12
abdication 13
able 14
about 15
above 16
abruptly 17
absolute 18
absorbed 19
absurdity 20


In [5]:
class Tokenizer():
    def __init__(self, vocab_dict):
        # Two dictionaries for changing the format of the data
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab_dict.items()}

    def encode(self, raw):

        # Return list of ids for any passed in raw text
        word_array = re.split(r'([_,.!-();:?"\']|--|\s)', raw)
        word_array = [item.lower() for item in word_array if item.strip()]

        # Handle unkept words
        word_array = [
            item if item in self.str_to_int
            else "<unk>" for item in word_array
        ]

        ids = [self.str_to_int[i] for i in word_array]
        return ids

    def decode(self, ids):
        # Convert list of ids back into text
        text = " ".join(self.int_to_str[i] for i in ids)
        text = re.sub(r'\s+([,.?!"\'])', r'\1', text)
        return text
        

In [6]:
# Lets run a quck demo on an instance of the tokenizer

tokenizer = Tokenizer(vocab)

text1 = "Hello, I am Neel!"
text2 = "How are you today?"
text = " <endoftext> ".join([text1, text2])
print(text)

Hello, I am Neel! <endoftext> How are you today?


In [7]:
tokenizer.encode(text)

[1076, 5, 481, 47, 1076, 0, 1077, 478, 66, 1072, 1076, 10]

In [8]:
tokenizer.decode(tokenizer.encode(text))

'<unk>, i am <unk>! <endoftext> how are you <unk>?'

In [9]:
import tiktoken
bpe_tokenizer = tiktoken.get_encoding('gpt2')

In [10]:
encoded_text = bpe_tokenizer.encode(raw)

In [11]:
# Let's create input-target pairs 

encoded_sample = encoded_text[50:]
context_size = 4
x = encoded_sample[:context_size]
y = encoded_sample[1:context_size+1] # Y is one more than x
print(x)
print(y)

[290, 4920, 2241, 287]
[4920, 2241, 287, 257]


In [12]:
import torch
from torch.utils.data import Dataset, DataLoader

class LLMDataset(Dataset):

    def __init__(self, text, tokenizer, length, stride):

        self.input_ids = []
        self.target_ids = []

        ids = tokenizer.encode(text)

        # Loop through data and append input and target tensors to the id arrays
        for i in range(0, len(ids) - length - 1, stride):

            input = ids[i:i+length]
            target = ids[i+1:i+1+length]

            self.input_ids.append(torch.tensor(input))
            self.target_ids.append(torch.tensor(target))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]

In [13]:
def create_dataloader(text, batch_size=4, length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):

    tokenizer = tiktoken.get_encoding('gpt2')

    dataset = LLMDataset(text, tokenizer, length, stride)

    dataloader = DataLoader(
        dataset, batch_size, shuffle, drop_last=drop_last, num_workers=num_workers
    )

    return dataloader

In [14]:
vocab_size = 50257
output_dim = 256

embd = torch.nn.Embedding(vocab_size, output_dim)

context_length = 4
dataloader = create_dataloader(raw, 8, context_length, stride=context_length, shuffle=False)

In [15]:
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [16]:
print(inputs)
print(inputs.shape)

tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
torch.Size([8, 4])


In [17]:
token_embeddings = embd(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [18]:
positional_embd_layer = torch.nn.Embedding(context_length, output_dim)
positional_embd = positional_embd_layer(torch.arange(context_length))

In [19]:
positional_embd.shape

torch.Size([4, 256])

In [20]:
input_embd = token_embeddings + positional_embd # Uses broadcasting to convert positional embd shape
input_embd.shape

torch.Size([8, 4, 256])

In [21]:
import torch.nn as nn

class SelfAttention(nn.Module):

    def __init__(self, dim_in, dim_out):
        super().__init__()

        # Initialize the query, key, and value weights
        self.W_query = nn.Linear(dim_in, dim_out, bias=False)
        self.W_key = nn.Linear(dim_in, dim_out, bias=False)
        self.W_value = nn.Linear(dim_in, dim_out, bias=False)

    def forward(self, x):
        # x shape: (batch_size, seq_len, num_tokens)
        
        query = self.W_query(x)
        keys = self.W_key(x)
        value = self.W_value(x)

        # Calculate the dot product between the query and key vectors
        attention_scores = query @ keys.T
        attention_weights = torch.softmax(attention_scores / (keys.shape[-1] ** 0.5), dim = -1)
        context_vector = attention_weights @ value

        return context_vector

In [22]:
print(input_embd.shape)
sample_input = input_embd[0]

torch.Size([8, 4, 256])


In [23]:
selfattention = SelfAttention(256, 6)
selfattention(sample_input)

tensor([[-1.1130, -0.0810,  1.2219, -0.6464, -1.0382, -0.8599],
        [-0.8514, -0.1273,  0.7553, -0.1253, -0.2789, -0.2997],
        [ 0.4808,  1.0377, -0.5203,  0.3048, -0.2056,  1.0742],
        [-0.7972, -0.0872,  0.9316, -0.1755, -0.4107, -0.3697]],
       grad_fn=<MmBackward0>)

In [24]:
# Trying to implement causal attention
class CasualSelfAttention(nn.Module):

    def __init__(self, dim_in, dim_out, context_length, dropout_val):
        super().__init__()

        self.dim_out = dim_out
        self.W_query = nn.Linear(dim_in, dim_out, bias=False)
        self.W_key = nn.Linear(dim_in, dim_out, bias=False)
        self.W_value = nn.Linear(dim_in, dim_out, bias=False)
        self.dropout = nn.Dropout(dropout_val)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        # x shape: (B, num tokens, vector emb size)
        batch, num_tokens, dim_in = x.shape

        queries = self.W_query(x) # B, num_tokens, d_out
        keys = self.W_key(x)
        values = self.W_value(x)
        # Take dot product between the query and keys
        # (B, num_tokens, d_out) x (B, d_out, num_tokens)
        attention_scores = queries @ keys.transpose(1,2) # (B, num_tokens, num_tokens)

        # Slice up self.mask to create causal mask for the correct num_tokens
        causal_mask = self.mask[:num_tokens, :num_tokens]
        attention_scores.masked_fill_(causal_mask == 1, -torch.inf)

        attention_weights = torch.softmax(attention_scores / (self.dim_out ** 0.5), dim=-1)
        attention_weights = self.dropout(attention_weights) # (B, num_tokens, dim_out)

        # Calculate context vector
        # (B, num_tokens, num_tokens) x (B, num_tokens, d_out)
        context_vector = attention_weights @ values # (B, num_tokens, d_out)
        return context_vector

In [25]:
input_embd = input_embd[0:2]

In [26]:
input_embd.shape


torch.Size([2, 4, 256])

In [27]:
causal_attention = CasualSelfAttention(256, 4, 4, 0.1)

In [28]:
print(causal_attention(input_embd))

tensor([[[ 0.0847,  0.3021, -0.4526, -0.6760],
         [ 0.6576, -0.4179, -0.7073, -0.3989],
         [-0.8214, -0.2609, -1.0635, -0.2402],
         [ 0.5318, -0.2881, -0.8292,  0.0809]],

        [[ 0.4292,  0.3867, -1.8741, -1.0178],
         [ 0.6667, -0.2257, -1.5472, -0.7061],
         [-0.1779, -0.1511, -0.4765,  0.1566],
         [ 0.1606, -0.2038, -0.9903, -0.2506]]], grad_fn=<UnsafeViewBackward0>)


In [29]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, num_heads, d_in, d_out, context_length, dropout_val):
        super().__init__()
        self.heads = [
            CasualSelfAttention(d_in, d_out, context_length, dropout_val)
            for _ in range(num_heads)
        ]

    def forward(self, x):

        return torch.cat([head(x) for head in self.heads], dim=-1)

In [30]:
class MultiHeadAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout_val, num_heads):

        super().__init__()

        # Ensure that the num_heads and d_out values are compatible
        assert(d_out % num_heads == 0)

        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)
        self.num_heads = num_heads
        self.d_out = d_out
        self.head_dim = d_out // num_heads
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))
        self.linear_layer = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout_val)

    def forward(self, x):

        # x: (B, num_tokens, d_in)
        batch, num_tokens, d_in = x.shape

        # (b, num_tokens, d_out)
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # Split into the heads
        keys = keys.view(batch, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(batch, num_tokens, self.num_heads, self.head_dim)
        values = values.view(batch, num_tokens, self.num_heads, self.head_dim)

        # Transpose so that shape is (batch, num_heads, num_tokens, head_dims)
        keys = keys.transpose(1,2)
        queries = queries.transpose(1,2)
        values = values.transpose(1,2)

        # Calculate the attention scores
        # (B, num_heads, num_tokens, head_dim) x (B, num_heads, head_dim, num_tokens)
        # (B, num_heads, num_tokens, num_tokens)
        attention_scores = queries @ keys.transpose(2,3) 

        # Create mask and apply
        causal_mask = self.mask[:num_tokens, :num_tokens]
        attention_scores.masked_fill_(causal_mask == 1, -torch.inf)

        attention_scores = attention_scores / (self.head_dim ** 0.5)
        attention_weights = torch.softmax(attention_scores, dim=-1) # Across
        attention_weights = self.dropout(attention_weights)
        # (B, num_heads, num_tokens, num_tokens) x (B, num_heads, num_tokens, head_dim)
        # (B, num_heads, num_tokens, head_dim)
        context_vector = attention_weights @ values

        # Reshape so that num_tokens is first
        context_vector = context_vector.transpose(1,2)
        context_vector = context_vector.contiguous().view(batch, num_tokens, self.d_out)
        context_vector = self.linear_layer(context_vector)

        return context_vector
        

In [31]:
input_embd.shape

torch.Size([2, 4, 256])

In [32]:
multiheadattn = MultiHeadAttention(256, 4, 4, 0.0, 2)

In [33]:
multiheadattn(input_embd)

tensor([[[-0.5132, -0.3560,  0.6204,  0.5386],
         [ 0.0831,  0.9459, -0.2315, -0.4558],
         [-0.2459,  0.6058,  0.0242, -0.4523],
         [ 0.1505,  0.6799, -0.0606, -0.5283]],

        [[-1.0462,  0.0524,  0.5643,  0.5261],
         [-1.0334,  0.3679,  0.3667,  0.2961],
         [-0.9032,  0.4218,  0.2965,  0.0014],
         [-0.9665,  0.3389,  0.3522, -0.2342]]], grad_fn=<ViewBackward0>)

In [34]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1 
}

In [35]:
class FeedForward(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(config['emb_dim'], 4*config['emb_dim']),
            nn.GELU(approximate='tanh'),
            nn.Linear(4*config['emb_dim'], config['emb_dim'])
        )

    def forward(self, x):
        return self.layers(x)

In [36]:
class TransformerBlock(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.attention = MultiHeadAttention(
            config['emb_dim'],
            config['emb_dim'],
            config['context_length'],
            config['drop_rate'],
            config['n_heads'],
        )
        self.forward_block = FeedForward(config)
        self.dropout = nn.Dropout(config['drop_rate'])
        self.layer_norm = nn.LayerNorm(config['emb_dim'])

    def forward(self, x):

        shortcut = x
        x = self.layer_norm(x)
        x = self.attention(x) # (b, # tokens, d_out)
        x = self.dropout(x)
        x = x + shortcut
 
        shortcut = x
        x = self.layer_norm(x)
        x = self.forward_block(x)
        x = self.dropout(x)
        x = x + shortcut

        return x

In [37]:
class GPTModel(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.tok_emb = nn.Embedding(config['vocab_size'], config['emb_dim'])
        self.pos_emb = nn.Embedding(config['context_length'], config['emb_dim'])
        self.drop = nn.Dropout(config['drop_rate'])

        self.transformer_blocks = nn.Sequential(
            *[TransformerBlock(config) for _ in range(config['n_layers'])]
        )

        self.final_layer_norm = nn.LayerNorm(config['emb_dim'])
        self.out_head = nn.Linear(
            config['emb_dim'], config['vocab_size'], bias=False
        )

        # self.out_head.weight = self.tok_emb.weight

    def forward(self, x):
        # (batch, seq_len)
        batch_size, sequence_length = x.shape 
        tok_embeds = self.tok_emb(x)
        pos_embeds = self.pos_emb(torch.arange(sequence_length, device=x.device))
        x = tok_embeds + pos_embeds
        x = self.drop(x)
        x = self.transformer_blocks(x)
        x = self.final_layer_norm(x)
        logits = self.out_head(x)
        return logits # (B, sequence_len, vocab_size)
        

In [38]:
# Sample input
model = GPTModel(GPT_CONFIG_124M)
batch = torch.rand((2, 4))
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

out = model(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [39]:
print(f"out shape: {out.shape}")

out shape: torch.Size([2, 4, 50257])


In [40]:
sum(p.numel() for p in model.parameters())

162401280

In [41]:
def generate_next_word(model, input, max_new_tokens, max_context_size):
    # Input (batch, num_tokens)

    for _ in range(max_new_tokens):

        # Crop the input to ensure its < max_context_size
        input_context = input[:, -max_context_size:]

        with torch.no_grad():
            logits = model(input_context)

        # logits (batch, num_tokens, vocab_size)
        # Focus on only the last row (all of the tokens being fed into the model)
        logits = logits[:, -1, :]
        probs = torch.softmax(logits, dim=-1)
        next_word_idx = torch.argmax(probs, dim=-1, keepdim=True)

        input = torch.cat((input, next_word_idx), dim=1)

    return input

In [42]:
start_context = "my name is"
encoded = tokenizer.encode(start_context)
print("encoded:", encoded)
encoded_tensor = torch.tensor(encoded).unsqueeze(0) #A
print("encoded_tensor.shape:", encoded_tensor.shape)

encoded: [1820, 1438, 318]
encoded_tensor.shape: torch.Size([1, 3])


In [43]:
model.eval()
output = generate_next_word(model, encoded_tensor, 6, GPT_CONFIG_124M['context_length'])

In [44]:
decoded = tokenizer.decode(output.squeeze().tolist())

In [45]:
print(decoded)

my name is Miner AAC constitutes Chall diamondshumans


In [46]:
def text_to_token_ids(text, tokenizer: tiktoken.Encoding):

    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    encoded_tensor = torch.tensor(encoded).unsqueeze(dim=0) # add batch
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer: tiktoken.Encoding):

    token_ids = token_ids.unsqueeze(0)
    return tokenizer.decode(token_ids.squeeze().tolist())

start_context = "that is so cool"

token_ids = generate_next_word(model, text_to_token_ids(start_context, tokenizer), 
                               10, GPT_CONFIG_124M['vocab_size'])

In [47]:
print(token_ids_to_text(token_ids, tokenizer))

that is so cool appears hun admittedly.— haircut clamp US Crab beneficiary shortstop


In [48]:
train_ratio = 0.90
split_idx = int(train_ratio * len(raw))
train_data = raw[0:split_idx]
val_data = raw[split_idx: ]

train_loader = create_dataloader(
    train_data,
    2,
    GPT_CONFIG_124M['context_length'],
    GPT_CONFIG_124M['context_length'],
    num_workers=0
)

val_loader = create_dataloader(
    val_data,
    2,
    GPT_CONFIG_124M['context_length'],
    GPT_CONFIG_124M['context_length'],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [49]:
for x,y in train_loader:
    print(x.shape, y.shape)
for x,y in val_loader:
    print(x.shape, y.shape)


torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])


In [50]:
def calculate_loss_batch(input_batch, target_batch, model, device):

    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0,1), target_batch.flatten())
    return loss

In [56]:
def calculate_loss_loader(train_loader, model, device, num_batches=len(train_loader)):

    total_loss = 0.0

    for i, (x_batch, y_batch) in enumerate(train_loader):

        if (i < num_batches):
            curr_loss = calculate_loss_batch(x_batch, y_batch, model, device)
            total_loss += curr_loss

    avg_loss = total_loss / num_batches
    return avg_loss

In [57]:
print(calculate_loss_loader(train_loader, model, 'cpu'))

tensor(10.9734, grad_fn=<DivBackward0>)


In [60]:
@torch.no_grad
def eval_model(model, val_loader, device, eval_iter):
    model.eval()
    val_loss = calculate_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return val_loss

In [61]:
@torch.no_grad
def generate_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    id_context = text_to_token_ids(start_context, tokenizer).to(device)
    token_ids = generate_next_word(model, id_context, 50, context_size)
    decoded = token_ids_to_text(token_ids, tokenizer)
    print(decoded.replace("\n", " "))
    model.train()


In [65]:
def train_model(model, train_loader, val_loader, optimizer, device, num_epochs,
                eval_freq, eval_iter, start_context, tokenizer):

    # Lists to track losses and tokens seen
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()
        running_train_loss = 0.0

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calculate_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()

            # Accumulate loss
            running_train_loss += loss.item()
            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                avg_train_loss = running_train_loss / (eval_freq)
                running_train_loss = 0.0

                val_loss = eval_model(model, val_loader, device, eval_iter)
                train_losses.append(avg_train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Epoch: {epoch+1} (Step: {global_step:06d}): "
                      f"Train Loss {avg_train_loss:.3f} val loss: {val_loss:.3f}")
            
        generate_sample(model, tokenizer, device, start_context)
    return train_losses, val_losses, track_tokens_seen


In [69]:
model = GPTModel(GPT_CONFIG_124M)
device = 'mps'
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)
num_epochs = 100
train_losses, val_losses, tokens_seen = train_model(model, train_loader, val_loader, optimizer, 
        device, num_epochs, 10, 10, "Every effort moves you", tokenizer)

Epoch: 1 (Step: 000000): Train Loss 1.096 val loss: 0.998
Every effort moves you the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the
Epoch: 2 (Step: 000010): Train Loss 1.416 val loss: 0.704
Every effort moves you, the, the, the, the the, the, the, the.                                  
Epoch: 3 (Step: 000020): Train Loss 1.815 val loss: 0.652
Every effort moves you, and, and, and, and, and, and the, and, and.                                
Epoch: 4 (Step: 000030): Train Loss 2.194 val loss: 0.639
Every effort moves you. Gisburn, and he was not to the picture. Gisburn. Gisburn's, and he was a to the picture. Gisburn, and he was--. I had the picture to the picture. I had to
Epoch: 5 (Step: 000040): Train Loss 2.378 val loss: 0.632
Every effort moves you know. "I I had been was "--and that, as I had been--and, and I had been  "--and.       "

KeyboardInterrupt: 